# Chuẩn hóa dữ liệu bằng scale()

Phương pháp: **scikit-learn `preprocessing.scale`**

Notebook đọc dữ liệu từ `github_repos_complete.csv`, chuẩn hóa các cột số được chỉ định và lưu kết quả ra file riêng. File gốc không bị ghi đè.

Hàm `sklearn.preprocessing.scale` trừ trung bình và chia cho độ lệch chuẩn của từng cột. Về mặt công thức, phương pháp này tương đương chuẩn hóa Z-score; notebook riêng này giúp bạn chạy và lưu kết quả theo đúng phương pháp/hàm `scale()`.

**Các cột được chuẩn hóa:**
- `star_count`
- `forks_count`
- `open_issues_count`
- `size_kb`


## 1. Đọc dữ liệu và kiểm tra cột

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_FILE = Path("github_repos_text_filled.csv")
OUTPUT_FILE = Path("github_repos_scale.csv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {INPUT_FILE.resolve()}. "
        "Hãy đặt CSV cùng thư mục với notebook hoặc sửa INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE, low_memory=False)

TARGET_COLUMNS = ['star_count', 'forks_count', 'open_issues_count', 'size_kb']
missing_columns = set(TARGET_COLUMNS) - set(df.columns)
if missing_columns:
    raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_columns}")

print(f"Số dòng: {len(df):,}")
print("Số giá trị thiếu ban đầu:")
display(df[TARGET_COLUMNS].isna().sum().to_frame("missing_count"))


Số dòng: 12,000
Số giá trị thiếu ban đầu:


,missing_count
star_count,0
forks_count,0
open_issues_count,0
size_kb,0


## 2. Chuẩn hóa các cột số

Chuyển các cột đích sang kiểu số. Giá trị không chuyển đổi được sẽ thành thiếu (`NaN`) và được giữ là thiếu khi chuẩn hóa.

In [2]:
# Ép kiểu số; giá trị không hợp lệ sẽ trở thành NaN
for column in TARGET_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

before_stats = df[TARGET_COLUMNS].describe().T
display(before_stats)


,count,mean,std,min,25%,50%,75%,max
star_count,12000.0,4657.808833,13516.041706,612.0,1065.00,1713.0,3483.00,393057.0
forks_count,12000.0,601.179750,2518.165190,0.0,88.00,174.0,392.00,107748.0
open_issues_count,12000.0,79.025167,596.567969,0.0,5.00,18.0,53.00,51103.0
size_kb,12000.0,109556.913917,889179.717088,0.0,2165.75,13518.5,58916.25,57192250.0


## 3. Áp dụng scikit-learn `preprocessing.scale`

Hàm `sklearn.preprocessing.scale` trừ trung bình và chia cho độ lệch chuẩn của từng cột. Về mặt công thức, phương pháp này tương đương chuẩn hóa Z-score; notebook riêng này giúp bạn chạy và lưu kết quả theo đúng phương pháp/hàm `scale()`.

**Quan trọng:** `scale()` không phải một công thức chuẩn hóa độc lập với Z-score; đây là cách triển khai Z-score trong scikit-learn.

In [3]:
from sklearn.preprocessing import scale

# sklearn.preprocessing.scale chuẩn hóa từng cột theo trung bình và độ lệch chuẩn
# tương đương về ý tưởng với Z-score (mean=0, std=1).
for column in TARGET_COLUMNS:
    values = df[column]
    non_missing = values.notna()
    if non_missing.sum() == 0:
        continue

    # Chỉ scale các giá trị không thiếu, giữ nguyên NaN
    df.loc[non_missing, column] = scale(values.loc[non_missing].to_numpy(dtype=float))

print("scale() hoàn tất. Lưu ý: sklearn.preprocessing.scale về bản chất là chuẩn hóa kiểu Z-score.")

scale() hoàn tất. Lưu ý: sklearn.preprocessing.scale về bản chất là chuẩn hóa kiểu Z-score.


C:\Users\7670\AppData\Local\Temp\ipykernel_18492\1966179570.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[11.25370374  6.40119334  4.10205166 ... -0.28802597 -0.28832193
 -0.28839592]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[non_missing, column] = scale(values.loc[non_missing].to_numpy(dtype=float))
C:\Users\7670\AppData\Local\Temp\ipykernel_18492\1966179570.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 8.82656407  1.22944645  3.28817398 ... -0.2264361  -0.22603897
 -0.17798611]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[non_missing, column] = scale(values.loc[non_missing].to_numpy(dtype=float))
C:\Users\7670\AppData\Local\Temp\ipykernel_18492\1966179570.py:12: FutureWarning: Setting an item of inc

## 4. Lưu file kết quả và kiểm tra

In [4]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu file: {OUTPUT_FILE.resolve()}")

print("Thống kê sau chuẩn hóa:")
display(df[TARGET_COLUMNS].describe().T)

print("Số giá trị thiếu sau chuẩn hóa:")
display(df[TARGET_COLUMNS].isna().sum().to_frame("missing_count"))

display(df[TARGET_COLUMNS].head(10))


Đã lưu file: D:\HomeWork\IE313\Đồ án\analysis-popularity-github-repository\notebooks\chuan_hoa\github_repos_scale.csv
Thống kê sau chuẩn hóa:


,count,mean,std,min,25%,50%,75%,max
star_count,12000.0,-1.421085e-17,1.000042,-0.299346,-0.265829,-0.217884,-0.086923,28.737361
forks_count,12000.0,-1.421085e-17,1.000042,-0.238747,-0.203800,-0.169646,-0.083072,42.551333
open_issues_count,12000.0,0.000000e+00,1.000042,-0.132472,-0.124090,-0.102298,-0.043627,85.532753
size_kb,12000.0,-6.513308e-18,1.000042,-0.123216,-0.120781,-0.108012,-0.056954,64.199701


Số giá trị thiếu sau chuẩn hóa:


,missing_count
star_count,0
forks_count,0
open_issues_count,0
size_kb,0


,star_count,forks_count,open_issues_count,size_kb
0,11.253704,8.826564,0.160885,0.454022
1,6.401193,1.229446,4.908237,0.129068
2,4.102052,3.288174,1.382926,0.216776
3,2.824405,3.296514,-0.092240,-0.120585
4,2.662369,0.930407,0.693956,-0.099623
5,2.126835,0.913330,-0.090564,-0.085849
6,1.883854,0.910947,0.129035,0.034714
7,1.523897,0.817224,-0.132472,0.207083
8,1.482019,0.100006,0.164238,-0.121898
9,1.454125,0.928024,0.164238,-0.108940


## Lưu ý
- Chỉ bốn cột nêu trên được biến đổi; các cột khác được giữ nguyên.
- Giá trị thiếu không được tự động điền.
- Các phép chuẩn hóa được fit trên toàn bộ file đầu vào. Nếu dùng dữ liệu cho mô hình học máy, hãy fit bộ chuẩn hóa chỉ trên tập train rồi áp dụng cho tập test để tránh data leakage.
